Hypothesis: the words of a caption carry signal beyond its length. This notebook does not load the test set at all, which keeps you honest.


In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, polars as pl
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().parent
P = ROOT / "data" / "processed"
tr = pl.read_parquet(P / "train_features.parquet").to_pandas()
va = pl.read_parquet(P / "validation_features.parquet").to_pandas()
for d in (tr, va):
    d["sound_log"] = np.log1p(d["sound_freq"])

CAT = ["country", "language", "type"]
NUM = ["hour", "weekday", "duration_ms", "caption_len", "n_hashtags",
       "n_mentions", "n_onscreen", "is_ad", "is_shop_video", "sound_log"]
FEATS = CAT + NUM
y_tr, y_va = tr["is_viral"], va["is_viral"]
cats = {c: sorted(tr[c].unique()) for c in CAT}

def prep(d):
    X = d[FEATS].copy().reset_index(drop=True)
    for c in CAT:
        X[c] = pd.Categorical(X[c].where(X[c].isin(cats[c])), categories=cats[c])
    return X

def fit_gbm(Xa, Xb):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31,
        min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, verbose=-1)
    m.fit(Xa, y_tr, eval_set=[(Xb, y_va)], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    return m

def report(name, y, p):
    k = int(len(y) * 0.10); top = np.argsort(-p)[:k]
    print(f"{name}: PR-AUC {average_precision_score(y, p):.3f} | ROC-AUC {roc_auc_score(y, p):.3f} | recall top 10% {y.values[top].sum()/y.sum():.3f}")

def paired_gain(p_new, p_old, n=500):
    rng = np.random.default_rng(0); out = []
    for _ in range(n):
        i = rng.integers(0, len(y_va), len(y_va)); yy = y_va.values[i]
        out.append(average_precision_score(yy, p_new[i]) - average_precision_score(yy, p_old[i]))
    lo, hi = np.percentile(out, [2.5, 97.5])
    print(f"gain {np.mean(out):.3f}, 95% interval [{lo:.3f}, {hi:.3f}]")

In [2]:
Xtr, Xva = prep(tr), prep(va)
gbm0 = fit_gbm(Xtr, Xva)
p0 = gbm0.predict_proba(Xva)[:, 1]
report("Reference LightGBM", y_va, p0)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[196]	valid_0's average_precision: 0.203731	valid_0's binary_logloss: 0.301156
Reference LightGBM: PR-AUC 0.204 | ROC-AUC 0.709 | recall top 10% 0.244


In [3]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5, sublinear_tf=True)
Ttr = tfidf.fit_transform(tr["caption"].fillna(""))
Tva = tfidf.transform(va["caption"].fillna(""))
txt = LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced").fit(Ttr, y_tr)
report("Text only", y_va, txt.predict_proba(Tva)[:, 1])

Text only: PR-AUC 0.145 | ROC-AUC 0.616 | recall top 10% 0.175


In [4]:
svd = TruncatedSVD(n_components=50, random_state=42)
cols = [f"svd_{i}" for i in range(50)]
Xtr2 = pd.concat([Xtr, pd.DataFrame(svd.fit_transform(Ttr), columns=cols)], axis=1)
Xva2 = pd.concat([Xva, pd.DataFrame(svd.transform(Tva), columns=cols)], axis=1)
gbm1 = fit_gbm(Xtr2, Xva2)
p1 = gbm1.predict_proba(Xva2)[:, 1]
report("LightGBM + text", y_va, p1)
paired_gain(p1, p0)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[158]	valid_0's average_precision: 0.198422	valid_0's binary_logloss: 0.301984
LightGBM + text: PR-AUC 0.198 | ROC-AUC 0.707 | recall top 10% 0.237
gain -0.005, 95% interval [-0.015, 0.005]


## Conclusion: caption text features

Hypothesis: the words of a caption carry signal beyond its length, hashtags and language.
Caption text alone reaches PR-AUC 0.145 (random 0.10), so the words do contain some signal.
However, adding a compressed TF-IDF representation to LightGBM gives PR-AUC 0.198 versus 0.204
for the reference model. The paired gain is -0.005 with a 95% interval of [-0.015, 0.005],
which includes 0 and is below the pre-set threshold of +0.01.
**Decision:** the text features are not kept. The information in the words is already captured
by caption length, number of hashtags and language.